In [1]:
import os

In [2]:
%pwd

'd:\\Somesh\\NLP_Project\\research'

In [3]:
from pathlib import Path
import os

# Locate the project root without assuming the notebook's launch directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = next(
    (
        path for path in candidates
        if (path / "config" / "config.yaml").exists()
        and (path / "src" / "NLP_Project").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Project root. Start Jupyter from "
        "D:/Somesh/NLP_Project or open this notebook inside that project."
    )

os.chdir(project_root)
print(f"Project root: {Path.cwd()}")


Project root: D:\Somesh\NLP_Project


In [4]:
%pwd

'D:\\Somesh\\NLP_Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str


In [6]:
from NLP_Project.constants import *
from NLP_Project.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=str(config.tokenizer_name),
        )


In [8]:
import os
from pathlib import Path

from NLP_Project.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk


d:\Somesh\NLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

        if not self.config.data_path.exists():
            raise FileNotFoundError(
                f"Input dataset path does not exist: {self.config.data_path}"
            )

        logger.info(f"Loading tokenizer: {self.config.tokenizer_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_name
        )

    def convert_examples_to_features(self, example_batch):
        model_inputs = self.tokenizer(
            example_batch["dialogue"],
            max_length=1024,
            truncation=True,
        )

        labels = self.tokenizer(
            text_target=example_batch["summary"],
            max_length=128,
            truncation=True,
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    def convert(self):
        logger.info(f"Loading dataset from: {self.config.data_path}")
        dataset = load_from_disk(str(self.config.data_path))

        logger.info("Tokenizing dataset...")
        tokenized_dataset = dataset.map(
            self.convert_examples_to_features,
            batched=True,
            desc="Tokenizing SAMSum dataset",
        )

        output_path = self.config.root_dir / "samsum_dataset"
        logger.info(f"Saving tokenized dataset to: {output_path}")
        tokenized_dataset.save_to_disk(str(output_path))

        logger.info("Data transformation completed successfully.")
        return tokenized_dataset


In [ ]:
# The ConfigurationManager and DataTransformation classes are defined above
# in this notebook. Do not import them from .py modules unless you have created
# those modules in src/NLP_Project.
from NLP_Project.logging import logger


In [10]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()

    data_transformation = DataTransformation(
        config=data_transformation_config
    )
    transformed_dataset = data_transformation.convert()

    print("Data transformation completed.")
    print(transformed_dataset)
except Exception:
    logger.exception("Data transformation failed.")
    raise


[2026-10-24 00:25:58,104: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-24 00:25:58,111: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-24 00:25:58,113: INFO: common: created directory at: artifacts]
[2026-10-24 00:25:58,116: INFO: common: created directory at: artifacts/data_transformation]
[2026-10-24 00:25:58,117: INFO: 2053974202: Loading tokenizer: google/pegasus-cnn_dailymail]
[2026-10-24 00:25:58,459: INFO: _client: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"]
[2026-10-24 00:25:58,697: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail "HTTP/1.1 200 OK"]
[2026-10-24 00:25:58,925: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-10-24 00:25:58,967: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-10-24 00:25:58,985: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"]
[2026-10-24 00:25:59,012: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Fconfig.json=&etag=%222c1a911e577525af99c26c1634c473667e1e7ae2%22 "HTTP/1.1 200 OK"]


d:\Somesh\NLP_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Somesh\.cache\huggingface\hub\models--google--pegasus-cnn_dailymail. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[2026-10-24 00:25:59,277: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-10-24 00:25:59,293: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Ftokenizer_config.json=&etag=%220a6130bfb0bfa44a3a376fe225b327e7ba8fa889%22 "HTTP/1.1 200 OK"]
[2026-10-24 00:25:59,316: INFO: _client: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json?%2Fgoogle%2Fpegasus-cnn_dailymail%2Fresolve%2F40d588fdab0cc077b80d950b300bf66ad3c75b92%2Ftokenizer_config.json=&etag=%220a6130bfb0bfa44a3a376fe225b327e7ba8fa889%22 "HTTP/1.1 200 OK"]
[2026-10-24 00:25

Tokenizing SAMSum dataset: 100%|██████████| 818/818 [00:00<00:00, 6588.76 examples/s]

[2026-10-24 00:26:08,991: INFO: 2053974202: Saving tokenized dataset to: artifacts\data_transformation\samsum_dataset]



Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 62669.02 examples/s]

[2026-10-24 00:26:09,127: INFO: 2053974202: Data transformation completed successfully.]
Data transformation completed.
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
})
